# [1교시]

In [25]:
import torch
import torch.nn as nn
import numpy as np

In [26]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

In [27]:
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}

In [28]:
import pandas as pd
df = pd.DataFrame(data)
df.head(1)

,reviews,ratings
0,이 영화 정말 좋아 최고야,1


In [29]:
# 전처리
import re
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text)
df['clean'] = df['reviews'].apply(lambda x : clean_data(x))

# 토큰분류 형태소
from konlpy.tag import Okt
okt = Okt()
def kor_tokened(text):    
    return [word for word, pos in okt.pos(text,stem=True) if len(word) > 1 and pos in ['Noun','Verb','Adjective']]

df['tokens'] = df['clean'].apply(lambda x : kor_tokened(x))

In [30]:
df

,reviews,ratings,clean,tokens
0,이 영화 정말 좋아 최고야,1,이 영화 정말 좋아 최고야,"[영화, 정말, 좋다, 최고]"
1,시간 아까운 쓰레기 영화,0,시간 아까운 쓰레기 영화,"[시간, 아깝다, 쓰레기, 영화]"
2,배우들 연기가 너무 훌륭해요,1,배우들 연기가 너무 훌륭해요,"[배우, 연기, 훌륭하다]"
3,스토리가 지루하고 뻔하다,0,스토리가 지루하고 뻔하다,"[스토리, 지루하다, 뻔하다]"
4,인생 최고의 명작입니다 추천,1,인생 최고의 명작입니다 추천,"[인생, 최고, 명작, 이다, 추천]"
5,돈 주고 보기 아까운 졸작,0,돈 주고 보기 아까운 졸작,"[주다, 보기, 아깝다, 졸작]"


# [2교시]

In [31]:
# 단어 인덱싱
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}
def make_vocab(tokens):
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)
df['tokens'].apply(lambda x : make_vocab(x))

0    None
1    None
2    None
3    None
4    None
5    None
Name: tokens, dtype: object

In [32]:
# padding   길이 맞추기
MAX_LEN = 5
def to_sequence(tokens):
    # 토큰화 - 패딩
    x = df['tokens'][4]
    x = [vocab.get(word, 1) for word in x]
    if len(x) < MAX_LEN:
        x = x + [0]*(MAX_LEN-len(x))
    else:
        x = x[:MAX_LEN]
    return x

df['pad_tokens'] = df['tokens'].apply(lambda x : to_sequence(x))
df

,reviews,ratings,clean,tokens,pad_tokens
0,이 영화 정말 좋아 최고야,1,이 영화 정말 좋아 최고야,"[영화, 정말, 좋다, 최고]","[15, 5, 16, 17, 18]"
1,시간 아까운 쓰레기 영화,0,시간 아까운 쓰레기 영화,"[시간, 아깝다, 쓰레기, 영화]","[15, 5, 16, 17, 18]"
2,배우들 연기가 너무 훌륭해요,1,배우들 연기가 너무 훌륭해요,"[배우, 연기, 훌륭하다]","[15, 5, 16, 17, 18]"
3,스토리가 지루하고 뻔하다,0,스토리가 지루하고 뻔하다,"[스토리, 지루하다, 뻔하다]","[15, 5, 16, 17, 18]"
4,인생 최고의 명작입니다 추천,1,인생 최고의 명작입니다 추천,"[인생, 최고, 명작, 이다, 추천]","[15, 5, 16, 17, 18]"
5,돈 주고 보기 아까운 졸작,0,돈 주고 보기 아까운 졸작,"[주다, 보기, 아깝다, 졸작]","[15, 5, 16, 17, 18]"


In [33]:
# 데이터셋, -- > Tensor타입변경 
# 데이터로더 --> 배치단위로 학습
from torch.utils.data  import Dataset, DataLoader
class ReviewMovieDataset(Dataset):
    def __init__(self,x,y):
        self.x = torch.LongTensor(x)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]
x = df['pad_tokens'].tolist();  y = df['ratings'].tolist()
dataset = ReviewMovieDataset(x,y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)        

In [34]:
import torch.nn.functional as F
class TextCNN(nn.Module):
    def __init__(self, vocab_size,embedding_dim, filter_sizes, num_fillers):
        super().__init__()        
        self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim=embedding_dim)
        # 합성곱 계층  텍스트는 이미지의 흑백처럼 채널을 1로 해서 사용
        # in_channel = 1(흑백이미지처럼 채널이 1)
        # kernel_size =(n-gram크기, 임베딩차원) -> 단어가 쪼개지지 않도록 width embedding dim과 동일
        self.convs =  nn.ModuleList(
                        [nn.Conv2d(in_channels=1, out_channels = num_fillers,
                                kernel_size=(fs, embedding_dim))
                                for fs in filter_sizes]
        )
        # 완전연결층,FC,분류기 입력
        # 추출된특징들을 모두 이어 붙임
        self.fc = nn.Linear(len(filter_sizes)*num_fillers, 1) # 이진분류
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        # x.shape (batch, seq_len)  (2, 5)
        # 임베딩 적용 -> (batch,seq_len,embedding_dim) (2,5,embedding_dim)
        x = self.embedding(x)
        # conv2d 입력(batch, channel, height, width) 채널정보를 받아야해서
        # shape : (batch,1,seq_len, embedding_dim)
        x = x.unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            # 합성곱, Relu 연산  shape : (batch,num_filters, seq_len-filter_size+1 ,1)
            c = F.relu(conv(x))  # shape(batch, num_filters,seq_len-filter_size+1)
            c = c.squeeze(3)
            # 최대폴링 : 문장에서 가장 강력한 특징 1개만 추출
            p = F.max_pool1d(c, c.size(2)) # shape (batch, num_filters,1)
            pooled_outputs.append(p.squeeze(2)) # shape (batch,num_filters)
        # 분류(추출된 피처들을 concatenate 후 FC레이어 통과)
        x_cat = torch.concatenate(pooled_outputs, dim=1) # shape(Batch, num_filters*len(filter_sizes))
        x_cat = self.dropout(x_cat)

        logits = self.fc(x_cat) # shape (Batch,1)
        return logits.squeeze(1) # shape (batch)

# [3교시]

In [35]:
# 모델 셋업
VOCAB_SIZE = len(vocab)
EMBED_DIM = 16 # 단어를 16차원 벡터로 표현
FILTER_SIZES = [2,3,4,5]  # 바이어그램(2), 트라이그램(3)사용
NUM_FILTERS = 10  # 각 사이즈별 필터 개수
model = TextCNN(VOCAB_SIZE,EMBED_DIM,FILTER_SIZES,NUM_FILTERS)

In [36]:
model

TextCNN(
  (embedding): Embedding(22, 16)
  (convs): ModuleList(
    (0): Conv2d(1, 10, kernel_size=(2, 16), stride=(1, 1))
    (1): Conv2d(1, 10, kernel_size=(3, 16), stride=(1, 1))
    (2): Conv2d(1, 10, kernel_size=(4, 16), stride=(1, 1))
    (3): Conv2d(1, 10, kernel_size=(5, 16), stride=(1, 1))
  )
  (fc): Linear(in_features=40, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [37]:
from torch.optim import Adam
criterion = nn.BCEWithLogitsLoss()  # 내부에 sigmoid 포함
optimizer = Adam(model.parameters(), lr=1e-3)

In [38]:
from tqdm import tqdm
epochs = 10
model.train()
for epoch in tqdm(range(epochs)):
    total_loss = 0.0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch : {epoch+1} / {epochs} loss : {( total_loss / len(dataloader) ):.4f}')

100%|██████████| 10/10 [00:00<00:00, 83.99it/s]

epoch : 1 / 10 loss : 0.6309
epoch : 2 / 10 loss : 0.7495
epoch : 3 / 10 loss : 0.7124
epoch : 4 / 10 loss : 0.6324
epoch : 5 / 10 loss : 0.7588
epoch : 6 / 10 loss : 0.6590
epoch : 7 / 10 loss : 0.6382
epoch : 8 / 10 loss : 0.6974
epoch : 9 / 10 loss : 0.6872
epoch : 10 / 10 loss : 0.7933


In [39]:
# 예측
model.eval()
test_review = '돈 주고 보기 아까운 졸작' # '이 영화 정말 좋아 최고야'
test_review = [
    vocab.get(word,1)
        for word in [word for word, pos in okt.pos(test_review,stem=True) if len(word) > 1]
]
if len(test_review) < MAX_LEN:
    test_review = test_review + [0]*(MAX_LEN - len(test_review))
test_review = torch.LongTensor(test_review)
test_review = test_review.unsqueeze(0)
with torch.no_grad():
    logits = model(test_review)
    prob = torch.sigmoid(logits)[0].item()
    print( '긍정' if prob > 0.5 else '부정' , prob)  

    

부정 0.4199974834918976


In [40]:
temp_x, temp_y = next(iter(dataloader))
temp_x.shape

torch.Size([2, 5])

# [4교시]

In [41]:
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [42]:
# rating 1~3  0   10~8  1  클래스 불균형 없게 40 : 40  --> 60
zero_mask = (1<=df['rating']) & (df['rating'] <=3)
zero_df_20 = df[zero_mask][:40]
zero_df_20['rating'] = 0


one_mask = (8<=df['rating']) & (df['rating'] <=10)
one_df_20 = df[one_mask][:40]
one_df_20['rating'] = 1

In [43]:
new_df = pd.concat((zero_df_20,one_df_20))
new_df

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,0,2018.10.29,인피니티 워
41,이건 뭐~ 이 정도 돈과 출연진 가지고 이렇게 망칠 수도 있구나를 보여주는 대표작...,0,2018.08.24,인피니티 워
49,중간중간 넘 지루해~~~~~~~~~~~,0,2018.08.16,인피니티 워
51,비젼보다 더 강한 스파이 ㅋㅋㅋㅋ,0,2018.08.13,인피니티 워
72,다잡은 보스를 쪼렙 한놈이 어이없는 열폭실수로 못죽여서 우주 생명체 50%가 죽는다...,0,2018.08.06,인피니티 워
...,...,...,...,...
53,"물론 타노스가 생명체의 절반을 날리는 데 성공한 것은 많이 아쉽지만, 그래도 정말 ...",1,2018.08.12,인피니티 워
54,결말이 다음 편을 위한 떡밥이라 만점주기는 머한... 그래도 이렇게 화려한 캐스팅을...,1,2018.08.12,인피니티 워
58,타노스는 이미 할일을 다 했다. 남은건 어벤져스가 과거로 돌아가 어느 부분에서 타...,1,2018.08.11,인피니티 워
60,당연히 10점 ㄱㄱ,1,2018.08.11,인피니티 워


In [44]:
# 1. 학습 테스트 데이터 분리
# 2. 데이터전처리(불용어,특수문자, 구두점 등등)  정규표현식
# 3. 토큰화 - 단어분리(okt 형태소별)
# 4. 단어사전 - 단어인덱스
# 5. 패딩    - 문서의 길이를 맞춤
# 6. 텐서변환 - 학습용 데이터 타입일치(torch tensor사용)
# 7. 데이터셋 + 데이터로더(학습용 테스용)
# 8. 모델셋팅(nn.Module상속받아서 클래스구성)  cnn, n-gram(필터-도장)  (n-gram, 스퀀스길이)
# 9. 하이퍼 파라메터 설정 - 옵티마이져, 러닝레이트, 손실함수 등등
# 10. 학습 - 데이터로더를 이용한 학습
# 11. 평가 - 데이터로드를 이용한 평가
# 12. 추론

In [45]:
# 1. 학습 테스트 데이터 분리
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(new_df['review'], new_df['rating'],test_size=0.2, random_state=42)

# 1. 데이터전처리
import re
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text).strip()

# 2. 토큰화
from konlpy.tag import Okt
okt = Okt()
def korea_tokenizer(text):
    return [
        word
    for word, pos in okt.pos(text,stem=True)
    if len(word) > 1 and pos in ['Noun','Verb','Adjective']
    ]
# 단어사전
vocab = {'<PAD>':0, '<UNK>' : 1}
def make_vocab(tokens):
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)


clean_data_tokens_list = []
for text in new_df['review'].tolist():
    clean_txt = clean_data(text)
    tokens = korea_tokenizer(clean_txt)
    if len(tokens) > 0:
        clean_data_tokens_list.append(tokens)


# clean_data_tokens_list = 
# [
#         korea_tokenizer(c_data) for c_data in 
#             [clean_data(text) for text in new_df['review'].tolist()] 
#     ]

for c_lists in clean_data_tokens_list: #[ ['좋아','영화'] ...  ]  
    if len(c_lists) > 0:
        make_vocab(c_lists)
# ---------  단어사전 완성 ------------------------

def review_pad_sequence(review,sequence_len):
    '''한글문장->전처리->한글토큰->단어사전을 이용한 인덱싱'''
    review = clean_data(review)
    review = [vocab.get(token,1) for token in korea_tokenizer(review)]        
    if len(review) < sequence_len:
        return review + [0]*(sequence_len - len(review))
    else:
        return review[:sequence_len]



# # 4. 텐서변환
# # 5. 데이터로더
from torch.utils.data import DataLoader, Dataset
class TextDataSet(Dataset):
    '''
        review_lists:List

        targets:List
    '''    
    def __init__(self,review_lists,targets,sequence_len=10):
        self.x = torch.LongTensor([ review_pad_sequence(review,sequence_len) for review in review_lists])
        self.y = torch.FloatTensor(targets)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]  
SEQ_LEN = 10    
x_train_dataset = TextDataSet(x_train.tolist(),y_train.tolist(),SEQ_LEN)
x_train_dataloader = DataLoader(x_train_dataset, batch_size=5, shuffle=True)

x_test_dataset = TextDataSet(x_test.tolist(),y_test.tolist(),SEQ_LEN)
x_test_dataloader = DataLoader(x_test_dataset, batch_size=5)

temp_x, temp_y = next(iter(x_train_dataloader))
temp_x.shape, temp_y.shape

(torch.Size([5, 10]), torch.Size([5]))

[ review_pad_sequence(review,10) for review in x_train.tolist()]

# [5교시] ~ [6교시]

In [46]:
# 모델은 상단에 정의된 클래스 이용
# 모델 셋업
VOCAB_SIZE = len(vocab)
EMBED_DIM = 16 # 단어를 16차원 벡터로 표현
FILTER_SIZES = [2,3,4,5]  # 바이어그램(2), 트라이그램(3)사용
NUM_FILTERS = 10  # 각 사이즈별 필터 개수
model = TextCNN(VOCAB_SIZE,EMBED_DIM,FILTER_SIZES,NUM_FILTERS)
optimizer = Adam(model.parameters(), lr=0.01)


from tqdm import tqdm
epochs = 47
model.train()
for epoch in tqdm(range(epochs)):
    total_loss = 0.0
    for batch_x, batch_y in x_train_dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions,batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch : {epoch+1} / {epochs} loss : {( total_loss / len(x_train_dataloader) ):.4f}')

  9%|▊         | 4/47 [00:00<00:02, 16.68it/s]

epoch : 1 / 47 loss : 0.7622
epoch : 2 / 47 loss : 0.7342
epoch : 3 / 47 loss : 0.5004
epoch : 4 / 47 loss : 0.4793


 17%|█▋        | 8/47 [00:00<00:02, 16.89it/s]

epoch : 5 / 47 loss : 0.4338
epoch : 6 / 47 loss : 0.2622
epoch : 7 / 47 loss : 0.2411
epoch : 8 / 47 loss : 0.2392


 28%|██▊       | 13/47 [00:00<00:01, 17.91it/s]

...

epoch : 37 / 47 loss : 0.0196
epoch : 38 / 47 loss : 0.0228
epoch : 39 / 47 loss : 0.0256
epoch : 40 / 47 loss : 0.0190


 91%|█████████▏| 43/47 [00:02<00:00, 15.85it/s]

epoch : 41 / 47 loss : 0.0433
epoch : 42 / 47 loss : 0.0183
epoch : 43 / 47 loss : 0.0186
epoch : 44 / 47 loss : 0.0190


100%|██████████| 47/47 [00:02<00:00, 16.44it/s]

epoch : 45 / 47 loss : 0.0074
epoch : 46 / 47 loss : 0.0076
epoch : 47 / 47 loss : 0.0113


In [47]:
# 평가
model.eval()
with torch.no_grad():
    total_loss = 0.0
    for batch_x, batch_y in x_test_dataloader:        
        print(batch_x,batch_y)
        predictions = model(batch_x)
        loss = criterion(predictions,batch_y)        
        total_loss += loss.item()
    print(f'{ (total_loss / len(x_test_dataloader)):.4f}')

tensor([[220, 105, 221,   3, 222, 223, 120, 224,   3,   0],
        [  2,   3,   4,   5,   0,   0,   0,   0,   0,   0],
        [ 16,  35, 172,  60,  37, 173, 174, 175, 105, 176],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [ 36,   0,   0,   0,   0,   0,   0,   0,   0,   0]]) tensor([0., 0., 0., 0., 0.])
tensor([[207, 208,   3, 209, 210, 211, 212, 105, 213, 134],
        [110,  36, 111, 112, 113, 114, 115, 116, 112, 117],
        [313,  53, 361, 257,  20, 105, 282,   3,  69, 362],
        [ 44,  45,  46,  47,  48,  49,  50,  51,  52,  53],
        [ 37, 139,   0,   0,   0,   0,   0,   0,   0,   0]]) tensor([0., 0., 1., 0., 0.])
tensor([[299,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [226, 107,   3,  37, 227, 228, 229,   0,   0,   0],
        [201,   3, 353, 354, 259,  20, 355, 117, 155, 312],
        [231, 141, 232,   0,   0,   0,   0,   0,   0,   0],
        [ 25, 189, 357, 279,  12,  73, 358,   0,   0,   0]]) tensor([1., 0., 1., 0., 1.])
tensor([[2

In [48]:
# 추론
test_x = '중간중간 넘 지루해'
test_x = review_pad_sequence(test_x,10)
test_x = torch.LongTensor(test_x)
test_x = test_x.unsqueeze(0)
logit = model(test_x)
prob = torch.sigmoid(logit).item()
prob

1.9823948605335318e-05

# [7교시]

In [49]:
# Gensim 라이브러리
# 대규모 한국어 코퍼스(Wikipedia, Naver News 등) 미리학습 Word2Vec, FastText 모델파일이 들어있는 라이브러리

In [51]:
# step1 : 사전학습된 모델 로드
import gensim
# FastText 모델 로드
# OOV(Out-Of-Vocabulary, 사전에 없는 단어) 문제에 강점이 있다, 오타가 많은 리뷰데이터에 특화

# step2 : 임베딩 행렬 구축
# 로드한 모델 수십만개의 단어 벡터가 있음, 우리가 구성한 단어장(vocab)에 있는 단어만 필요(단어장 크기, 임베딩 차원)크기의 빈행렬생성
# 우리 단어장에 있는 단어가 사전학습한 모델에 존재하면 그 벡터를 복사

# step3 : nn.Embedding에 가중치 덮어쓰기

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}
df = pd.DataFrame(data)

def tokenizer(text) : return text.split()

vocab = {'<PAD>':0, '<UNK>':1}
for review in df['reviews']:
    for word in tokenizer(review):
        if word not in vocab: vocab[word] = len(vocab)

max_sequence_len = 8
def text_to_sequence(text,vocab,max_sequence_len):
    seq = [vocab.get(word,vocab['<UNK>']) for word in tokenizer(text)]
    return seq + [vocab['<PAD>']]*(max_sequence_len-len(seq)) \
                if len(seq) < max_sequence_len \
                    else seq[:max_sequence_len]
df['sequences'] = df['reviews'].apply(lambda x : text_to_sequence(x,vocab,max_sequence_len))

class ReviewDataSet(Dataset):
    def __init__(self,sequence,labels):
        self.x = torch.LongTensor(sequence)
        self.y = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]
dataloader = DataLoader(ReviewDataSet(df['sequences'].tolist(), df['ratings'].tolist()),
                        batch_size=2,shuffle=True
                        )    

https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ko.300.vec.gz

In [55]:
import urllib
import gzip
import shutil
import os

print('파일 다운로드....')
url = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ko.300.vec.gz'
gz_filename = 'cc.ko.300.vec.gz'
vec_filename = 'cc.ko.300.vec'

urllib.request.urlretrieve(url, gz_filename)

print('압축해제......')
with gzip.open(gz_filename,'rb') as f_in:
    with open(vec_filename,'wb') as f_out:
        shutil.copyfileobj(f_in,f_out)

os.remove(gz_filename)
print('압축해제 및 파일정리 완료')

파일 다운로드....
압축해제......
압축해제 및 파일정리 완료


In [59]:
# 모델 로드(로컬 FastText) & 임베딩 행렬 구축
import gensim
VOCAB_SIZE = len(vocab)
EMBED_DIM = 300  # FastText가 300차원
MODEL_PATH = vec_filename
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(MODEL_PATH,binary=False, unicode_errors='replace')

In [60]:
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
oov_count = 0

for word, idx in vocab.items():
    if idx == 0: continue # <PAD>는 0 유지
    
    if word in w2v_model:
        embedding_matrix[idx] = w2v_model[word]
    else:
        # 사전에 없는 단어는 정규분포를 따르는 난수로 초기화
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(EMBED_DIM, ))
        oov_count += 1
        
print(f"-> 모델 로드 완료! (우리 단어장 단어: {VOCAB_SIZE}개 / 사전에 없는 단어: {oov_count}개)")
pretrained_weight = torch.tensor(embedding_matrix, dtype=torch.float32)

# ==========================================
# 4. TextCNN 모델 정의
# ==========================================
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, filter_sizes, num_filters, pretrained_weight=None, freeze_emb=False):
        super(TextCNN, self).__init__()
        
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # FastText 가중치 덮어씌우기
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(pretrained_weight)
            self.embedding.weight.requires_grad = not freeze_emb

        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embed_dim)) for fs in filter_sizes
        ])
        
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x)).squeeze(3)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            pooled_outputs.append(p)
            
        x_cat = self.dropout(torch.cat(pooled_outputs, dim=1))
        return self.fc(x_cat).squeeze(1)

# ==========================================
# 5. 모델 학습 (Training)
# ==========================================
FILTER_SIZES = [2, 3, 4] 
NUM_FILTERS = 100        

model = TextCNN(
    vocab_size=VOCAB_SIZE, 
    embed_dim=EMBED_DIM, 
    filter_sizes=FILTER_SIZES, 
    num_filters=NUM_FILTERS, 
    pretrained_weight=pretrained_weight, 
    freeze_emb=False # 학습하며 임베딩도 우리 데이터에 맞게 미세조정
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

print("\n--- 본격적인 학습을 시작합니다 ---")
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    model.train() 
    
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

print("--- 학습 완료! ---")

-> 모델 로드 완료! (우리 단어장 단어: 25개 / 사전에 없는 단어: 1개)

--- 본격적인 학습을 시작합니다 ---
Epoch 01/10 | Loss: 0.6968
Epoch 02/10 | Loss: 0.6229
Epoch 03/10 | Loss: 0.5246
Epoch 04/10 | Loss: 0.4452
Epoch 05/10 | Loss: 0.3930
Epoch 06/10 | Loss: 0.3327
Epoch 07/10 | Loss: 0.3016
Epoch 08/10 | Loss: 0.2084
Epoch 09/10 | Loss: 0.1813
Epoch 10/10 | Loss: 0.1467
--- 학습 완료! ---


## Word2Vec

In [ ]:
# 데이터 늘리기
# 한글 토크나이져 사용
# tfidfvectors도 사용
# 목표 ... 다음리뷰의 데이터로 train test 분리해서 최대한 test의 감정분류 높여보기
# 모델에서 cnn대신 RNN LSTM도 적용해서 각 모델별 성능의 차이

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from gensim.models import Word2Vec  # 사전 학습된 모델

# ==========================================
# 1. 데이터 준비
# ==========================================

data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [5, 1, 4, 2, 5, 1]
}

df = pd.DataFrame(data)

# 평점 이진화
# 4~5점 -> 긍정(1)
# 1~3점 -> 부정(0)

df['ratings'] = df['ratings'].apply(
    lambda x: 1 if x >= 4 else 0
)
# print(df)
# ==========================================
# 2. 토큰화
# ==========================================
def tokenize(text):
    return text.split()

tokenized_sentences = [
    tokenize(review)
    for review in df['reviews']
]

# print(tokenized_sentences)
# ==========================================
# 3. Word2Vec 사전학습 (전이학습용)
# ==========================================
EMBED_DIM = 100
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBED_DIM,
    window=3,
    min_count=1,
    workers=4
)
print("Word2Vec 사전학습 완료")

# ==========================================
# 4. Vocabulary 생성
# ==========================================
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}
for sentence in tokenized_sentences:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

# print(vocab)
VOCAB_SIZE = len(vocab)
# ==========================================
# 5. 문장 -> 숫자 변환
# ==========================================
MAX_LEN = 10
def text_to_sequence(text, vocab, max_len):
    seq = [
        vocab.get(word, vocab['<UNK>'])
        for word in tokenize(text)
    ]
    # padding
    if len(seq) < max_len:
        seq += [vocab['<PAD>']] * (max_len - len(seq))
    return seq[:max_len]

df['sequence'] = df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, MAX_LEN)
)

# print(df['sequence'])

# ==========================================
# 6. Embedding Matrix 생성
# ==========================================
embedding_matrix = np.random.normal(
    scale=0.01,
    size=(VOCAB_SIZE, EMBED_DIM)
)

# PAD 벡터는 0으로 유지
embedding_matrix[0] = np.zeros((EMBED_DIM,))

for word, idx in vocab.items():
    if word in ['<PAD>', '<UNK>']:
        continue

    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

pretrained_weight = torch.FloatTensor(
    embedding_matrix
)
# print(pretrained_weight.shape)

# ==========================================
# 7. Dataset
# ==========================================
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.LongTensor(sequences)
        self.y = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = ReviewDataset(
    df['sequence'].tolist(),
    df['ratings'].tolist()
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

# ==========================================
# 8. TextCNN 모델
# ==========================================

class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        filter_sizes,
        num_filters,
        pretrained_weight=None,
        freeze_emb=False
    ):
        super().__init__()
        # 전이학습 Embedding
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # pretrained weight 적용
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(
                pretrained_weight
            )
            # freeze 여부
            self.embedding.weight.requires_grad = (
                not freeze_emb
            )

        # CNN
        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels=1,
                out_channels=num_filters,
                kernel_size=(fs, embed_dim)
            )
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(
            len(filter_sizes) * num_filters,
            1
        )

    def forward(self, x):
        # (batch, seq_len)
        x = self.embedding(x)
        # (batch, seq_len, embed_dim)
        x = x.unsqueeze(1)
        # (batch, 1, seq_len, embed_dim)
        pooled_outputs = []
        for conv in self.convs:
            # convolution
            c = F.relu(conv(x))
            # (batch, num_filters, H, 1)
            c = c.squeeze(3)
            # (batch, num_filters, H)
            # max pooling
            p = F.max_pool1d(
                c,
                kernel_size=c.size(2)
            )
            p = p.squeeze(2)
            pooled_outputs.append(p)

        # concat
        x = torch.cat(
            pooled_outputs,
            dim=1
        )

        x = self.dropout(x)
        logits = self.fc(x)
        return logits.squeeze(1)

# ==========================================
# 9. 모델 생성
# ==========================================

FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 20

model = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    filter_sizes=FILTER_SIZES,
    num_filters=NUM_FILTERS,
    pretrained_weight=pretrained_weight,
    freeze_emb=False
)

print(model)

# ==========================================
# 10. Loss / Optimizer
# ==========================================

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ==========================================
# 11. 학습
# ==========================================

EPOCHS = 10

print("\n학습 시작\n")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(
            logits,
            batch_y
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(
        f"Epoch: {epoch+1:02d} "
        f"Loss: {avg_loss:.4f}"
    )

print("\n학습 완료")

# ==========================================
# 12. 추론
# ==========================================

def predict(text):
    model.eval()
    sequence = text_to_sequence(
        text,
        vocab,
        MAX_LEN
    )

    x = torch.LongTensor(sequence).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits)
        pred = (prob >= 0.5).float()

    print(f"\n리뷰: {text}")
    print(f"긍정확률: {prob.item():.4f}")

    if pred.item() == 1:
        print("예측: 긍정 ")
    else:
        print("예측: 부정 ")

Word2Vec 사전학습 완료
TextCNN(
  (embedding): Embedding(25, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv2d(1, 20, kernel_size=(2, 100), stride=(1, 1))
    (1): Conv2d(1, 20, kernel_size=(3, 100), stride=(1, 1))
    (2): Conv2d(1, 20, kernel_size=(4, 100), stride=(1, 1))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=60, out_features=1, bias=True)
)

학습 시작

Epoch: 01 Loss: 0.6924
Epoch: 02 Loss: 0.6878
Epoch: 03 Loss: 0.6836
Epoch: 04 Loss: 0.6866
Epoch: 05 Loss: 0.6864
Epoch: 06 Loss: 0.6946
Epoch: 07 Loss: 0.6839
Epoch: 08 Loss: 0.6798
Epoch: 09 Loss: 0.6819
Epoch: 10 Loss: 0.6691

학습 완료


In [62]:
# ==========================================
# 13. 테스트
# ==========================================

predict("배우 연기가 정말 훌륭하다")
predict("돈 아까운 최악의 영화")


리뷰: 배우 연기가 정말 훌륭하다
긍정확률: 0.4930
예측: 부정 

리뷰: 돈 아까운 최악의 영화
긍정확률: 0.4835
예측: 부정 
